# My julia implementation

# Install Instructions

I will assume that you have Julia installed on your local machine. (If not go to the Julia website: https://julialang.org/ and use the "Download" button.)

### 1. Install IJulia

I could put instructions here but it is probably better to just look at the github readme here: https://github.com/JuliaLang/IJulia.jl or the installation instructions in the documentation here: https://julialang.github.io/IJulia.jl/stable/manual/installation/

### 2. Launch the Notebook and Run

Launch the notebook and run cells as you would any other jupyter notebook.

NOTE: The first cell below will create a Julia environment in the same directory as this notebook resides in. The second cell will install the needed packages.

Progressive Hedging Repo: https://github.com/NREL/ProgressiveHedging.jl
Other JuMP Compatible Optimizers: https://jump.dev/JuMP.jl/stable/installation/#Install-a-solver

In [1]:
import Pkg
Pkg.activate(".")
Pkg.status()

      Status `~/Desktop/projects/Quantum-Stochastic-Optimization/Project.toml`
  [87dc4568] HiGHS v1.7.3
  [b6b21f68] Ipopt v1.5.0
  [4076af6c] JuMP v1.15.1


  Activating project at `~/Desktop/projects/Quantum-Stochastic-Optimization`


In [2]:
#Pkg.add("JuMP")
#Pkg.add("HiGHS")
#Pkg.add("Ipopt")

In [3]:
import HiGHS
import Ipopt
import JuMP

In [14]:
# MY VERSION
#c_x = 3.0  # single cost for integer x
c_w = 1.0
c_y = 10.0
thermal_min = 0
thermal_max = 3
wind_min = 0
wind_max = 3
# wind_set = collect(0:3)
d = 3
ξ = collect(wind_min:1:wind_max)
NSCEN = length(ξ)

NGEN = 3
c_x = [4.,2.,3.]  # different fixed and operational costs for generators 

prob = [0.1,0.2,0.3,0.4]
#prob = [0.25, 0.25, 0.0, 0.5] # prob[s] gives probability of ξ taking value ξ[s]
# prob = ones(NSCEN) ./ NSCEN # Uniform (discrete) probabilities

4-element Vector{Float64}:
 0.1
 0.2
 0.3
 0.4

In [15]:
## Optimization Problem

# Create model
m = JuMP.Model(HiGHS.Optimizer)

# add generator variable
JuMP.@variable(m, x[1:NGEN], Bin)  # replace integer variable with a list of binary variables

# Add Variables
#JuMP.@variable(m, thermal_min <= x <= thermal_max, Int)
JuMP.@variable(m, wind_min <= w[1:NSCEN] <= wind_max, Int)
JuMP.@variable(m, 0 <= y_minus[1:NSCEN] <= d, Int)
;

# Add Constraints
JuMP.@constraint(m, sum(x) .+ w .+ y_minus .== d)
JuMP.@constraint(m, w .<= ξ)
;

# Add Objective
obj_expr = zero(JuMP.AffExpr)

#JuMP.add_to_expression!(obj_expr, c_x, x) # Thermal cost
for g in 1:NGEN
    JuMP.add_to_expression!(obj_expr, c_x[g], x[g]) # Thermal cost
end

for s in 1:NSCEN
    JuMP.add_to_expression!(obj_expr, prob[s]*c_w, w[s]) # Wind cost -- c_w == 0
    JuMP.add_to_expression!(obj_expr, prob[s]*c_y, y_minus[s]) # Lost load
end
JuMP.@objective(m, Min, obj_expr)
;

In [16]:
println(m)
;

Min 4 x[1] + 2 x[2] + 3 x[3] + 0.1 w[1] + y_minus[1] + 0.2 w[2] + 2 y_minus[2] + 0.3 w[3] + 3 y_minus[3] + 0.4 w[4] + 4 y_minus[4]
Subject to
 x[1] + x[2] + x[3] + w[1] + y_minus[1] = 3
 x[1] + x[2] + x[3] + w[2] + y_minus[2] = 3
 x[1] + x[2] + x[3] + w[3] + y_minus[3] = 3
 x[1] + x[2] + x[3] + w[4] + y_minus[4] = 3
 w[1] ≤ 0
 w[2] ≤ 1
 w[3] ≤ 2
 w[4] ≤ 3
 w[1] ≥ 0
 w[2] ≥ 0
 w[3] ≥ 0
 w[4] ≥ 0
 y_minus[1] ≥ 0
 y_minus[2] ≥ 0
 y_minus[3] ≥ 0
 y_minus[4] ≥ 0
 w[1] ≤ 3
 w[2] ≤ 3
 w[3] ≤ 3
 w[4] ≤ 3
 y_minus[1] ≤ 3
 y_minus[2] ≤ 3
 y_minus[3] ≤ 3
 y_minus[4] ≤ 3
 w[1] integer
 w[2] integer
 w[3] integer
 w[4] integer
 y_minus[1] integer
 y_minus[2] integer
 y_minus[3] integer
 y_minus[4] integer
 x[1] binary
 x[2] binary
 x[3] binary



In [17]:
## Solution

JuMP.optimize!(m)
;

Running HiGHS 1.6.0: Copyright (c) 2023 HiGHS under MIT licence terms
Presolving model
3 rows, 9 cols, 15 nonzeros
2 rows, 7 cols, 10 nonzeros
Objective function is integral with scale 10

Solving MIP model with:
   2 rows
   7 cols (4 binary, 3 integer, 0 implied int., 0 continuous)
   10 nonzeros

        Nodes      |    B&B Tree     |            Objective Bounds              |  Dynamic Constraints |       Work      
     Proc. InQueue |  Leaves   Expl. | BestBound       BestSol              Gap |   Cuts   InLp Confl. | LpIters     Time

         0       0         0   0.00%   4.2             inf                  inf        0      0      0         0     0.0s
 T       0       0         0   0.00%   4.2             6.9               39.13%        0      0      0         2     0.0s

Solving report
  Status            Optimal
  Primal bound      6.9
  Dual bound        6.9
  Gap               0% (tolerance: 0.01%)
  Solution status   feasible
                    6.9 (objective)
           

In [18]:
# @show JuMP.termination_status(m)
# @assert JuMP.termination_status(m) == JuMP.OPTIMAL # Throw an error if solve failed
# @show JuMP.objective_value(m)
# ;
JuMP.solution_summary(m)

for var in JuMP.all_variables(m)
    println(var, " = ", JuMP.value(var))
end

x[1] = 0.0
x[2] = 1.0
x[3] = 1.0
w[1] = 0.0
w[2] = 1.0
w[3] = 1.0
w[4] = 1.0
y_minus[1] = 1.0
y_minus[2] = 0.0
y_minus[3] = 0.0
y_minus[4] = 0.0


# Solution with x as an integer

In [8]:
# @show JuMP.termination_status(m)
# @assert JuMP.termination_status(m) == JuMP.OPTIMAL # Throw an error if solve failed
# @show JuMP.objective_value(m)
# ;
JuMP.solution_summary(m)

for var in JuMP.all_variables(m)
    println(var, " = ", JuMP.value(var))
end

x = 2.0
w[1] = 0.0
w[2] = 1.0
w[3] = 1.0
w[4] = 1.0
y_minus[1] = 1.0
y_minus[2] = 0.0
y_minus[3] = 0.0
y_minus[4] = 0.0
